<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SuGaR — Surface-Aligned Gaussian Splatting to Mesh (INRIA non-commercial)

**License warning:** SuGaR uses the [INRIA Gaussian-Splatting License](https://github.com/Anttwo/SuGaR/blob/main/LICENSE.md) — free for academic/industrial research and evaluation, **NOT for commercial use** without explicit INRIA consent. This notebook is included in AEI-Colab-Notebooks for **research and personal-asset evaluation only**. If you ship a commercial product, you must (a) get INRIA's permission, (b) remove SuGaR and use a commercial alternative (Kiri Engine, Polycam, etc.), or (c) train your own surface-aligned Gaussian method. See the **License** section in Step 1.

[SuGaR](https://github.com/Anttwo/SuGaR) ([Guédon & Lepetit, CVPR 2024](https://arxiv.org/abs/2311.12775)) by **Antoine Guédon and Vincent Lepetit** at INRIA, EPFL. Surface-Aligned Gaussian Splatting adds a **surface-alignment constraint** to vanilla 3D Gaussian Splatting training, then extracts a high-quality mesh from the resulting "surface-aligned Gaussians". The mesh quality is significantly better than naive point-cloud Poisson reconstruction (TripoSplat's default), at the cost of a much longer pipeline (vanilla 3DGS train + SuGaR alignment + mesh extraction).

**Why use this instead of TripoSplat's built-in mesh export?** TripoSplat's `*_mesh.ply` / `.glb` / `.fbx` outputs come from sampling 1 point per Gaussian and running Poisson reconstruction — fast but lossy, with the typical "balloon" artifacts and missing surfaces. SuGaR re-trains the Gaussians with a surface constraint, then extracts a mesh that follows the actual surface much more accurately. Trade-off: ~2-3 hours per scene on L4, vs ~30s for TripoSplat's default.

**What this notebook does:**

- **Step 1** — install pinned PyTorch 2.0.1+cu118 + pytorch3d 0.7.4 + clone SuGaR + build CUDA submodules (mandatory version pinning; mismatches are the #1 source of failures).
- **Step 2** — **TripoSplat-PLY-to-3DGS-scene bridge**: takes a single image + TripoSplat-generated 3DGS PLY and builds a minimal COLMAP-compatible scene directory with one camera (estimated from TripoSplat's preprocessing). SuGaR's design assumes multi-view; this bridge gives it something to chew on for single-image input. Quality gain over TripoSplat's default mesh will be **modest for single-image** (no multi-view consistency signal) and **dramatic for multi-image** (you'll get the best results if you have 3+ overlapping views of the same subject).
- **Step 3** — train vanilla 3DGS for 7k iterations (~30 min on L4) using SuGaR's bundled `gaussian_splatting/train.py`.
- **Step 4** — run SuGaR's surface-alignment and mesh extraction (`train.py -c ... --low_poly True --refinement_time short`).
- **Step 5** — verify outputs (textured `.obj` + texture atlas `.png` + refined 3DGS `.ply`).
- **Step 6** — convert `.obj` → `.glb` + apply game-asset transform (scale to meters, base-pivot center).
- **Step 7** — copy outputs to Drive at `/content/drive/MyDrive/AEI_3D_Out/SuGaR/`.
- **Step 8** — keep-alive (anti-disconnect).

**Compute:** L4 22 GB recommended. A100 40 GB gives ~40% speedup. T4 16 GB will likely OOM during 3DGS training. CPU-only is not viable (the 3DGS rasterizer is CUDA-only). First run: ~10-15 min install + ~2-3 hrs compute. Subsequent runs: ~10 min install + ~2-3 hrs compute.

In [ ]:
# @title STEP 1 — Install Dependencies (Pinned PyTorch 2.0.1 + cu118)
# @markdown Installs the exact dependency stack SuGaR needs. Pinned versions
# @markdown are mandatory — mismatches are the #1 source of failures (see SuGaR
# @markdown GitHub issue #163, `libc10_cuda.so` import error). If you have a
# @markdown Colab with a newer PyTorch preinstalled, this cell will swap to
# @markdown the right versions before installing pytorch3d + SuGaR's CUDA submodules.
#
# @markdown **Compute requirement:** CUDA-capable GPU with **16+ GB VRAM**.
# @markdown L4 (22 GB) is the sweet spot. A100 works too. T4 16 GB will likely
# @markdown OOM during vanilla 3DGS training.

import os, sys, subprocess, time

print('[sugar] SuGaR + INRIA license notice')
print('[sugar] Free for research/evaluation. NOT for commercial use without INRIA consent.')
print('[sugar] See: https://github.com/Anttwo/SuGaR/blob/main/LICENSE.md')
print('[sugar] If you ship a commercial product, do NOT use this notebook.')
print(flush=True)

REPO_DIR = '/content/SuGaR'
REPO_URL = 'https://github.com/Anttwo/SuGaR.git'
if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} (with submodules)...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules', REPO_URL, REPO_DIR], check=True)
    print(f'[git] Cloned to {REPO_DIR}', flush=True)
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.', flush=True)
    # Always re-fetch submodules in case they were skipped previously
    subprocess.run(['git', '-C', REPO_DIR, 'submodule', 'update', '--init', '--recursive'], check=False)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Pinned PyTorch 2.0.1 + CUDA 11.8 swap (MANDATORY) ──────────────────────
# Colab 2026.01 ships PyTorch 2.9 / cu126 by default. SuGaR was developed
# against PyTorch 2.0.1 / cu118 and the bundled CUDA submodules
# (diff-gaussian-rasterization, simple-knn) will not compile or run against
# newer CUDA. We MUST swap to the pinned versions BEFORE installing
# pytorch3d or building the submodules.
t0 = time.time()
print('[pip] Installing pinned PyTorch 2.0.1+cu118 + torchvision 0.15.2 ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'torch==2.0.1+cu118', 'torchvision==0.15.2+cu118', 'torchaudio==2.0.2+cu118',
                '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
print(f'[pip] torch swap: {time.time()-t0:.1f}s', flush=True)

# ── SuGaR's Python deps ──────────────────────────────────────────────
t0 = time.time()
print('[pip] Installing SuGaR Python deps ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'fvcore', 'iopath', 'plotly', 'rich',
                'plyfile==0.8.1', 'pymcubes', 'pyquaternion',
                'scikit-learn', 'pandas', 'tqdm', 'configargparse',
                'trimesh', 'open3d==0.17.0', 'gradio==5.49.1'],
               check=True)
print(f'[pip] deps: {time.time()-t0:.1f}s', flush=True)

# ── pytorch3d (unofficial cu118 wheel) ──────────────────────────────────────────
# pytorch3d 0.7.4 must match PyTorch 2.0.1 + cu118. We use the unofficial
# prebuilt wheel from Sebastian Raschka's PyTorch3D wheels collection.
# If the wheel URL changes, this is the most likely failure point — fall
# back to `conda install pytorch3d=0.7.4` in a conda env if pip fails.
t0 = time.time()
print('[pip] Installing pytorch3d 0.7.4 (cu118 wheel)...', flush=True)
PYTORCH3D_WHEEL = 'https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu118_pyt201/pytorch3d-0.7.4-cp310-cp310-linux_x86_64.whl'
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', PYTORCH3D_WHEEL], check=False)
if result.returncode != 0:
    print(f'[pip] pytorch3d wheel install failed (returncode={result.returncode})', flush=True)
    print('[pip] Falling back to building pytorch3d from source (slow but works)...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch3d==0.7.4'], check=True)
print(f'[pip] pytorch3d: {time.time()-t0:.1f}s', flush=True)

# ── Build SuGaR's CUDA submodules (MANDATORY, slow) ─────────────────────────
# diff-gaussian-rasterization and simple-knn are CUDA extensions that
# MUST be compiled against the installed PyTorch. Building takes 2-5 min.
for sub in ('gaussian_splatting/submodules/diff-gaussian-rasterization',
            'gaussian_splatting/submodules/simple-knn'):
    if not os.path.isdir(sub):
        print(f'[build] WARNING: {sub} not present (submodule fetch failed)', flush=True)
        continue
    t0 = time.time()
    print(f'[build] Compiling {sub} (this takes 1-3 min) ...', flush=True)
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', sub], check=False)
    if result.returncode == 0:
        print(f'[build] {sub}: OK ({time.time()-t0:.1f}s)', flush=True)
    else:
        print(f'[build] {sub}: FAILED (returncode={result.returncode})', flush=True)
        print(f'[build] This is the #1 source of SuGaR failures.', flush=True)
        print(f'[build] Check that PyTorch==2.0.1+cu118 is installed and try again.', flush=True)

# ── nvdiffrast (optional but recommended — speeds up textured mesh) ─────────────────
print('[pip] Installing nvdiffrast (optional, speeds up texture extraction)...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvdiffrast'], check=False)

# ── Sanity check ─────────────────────────────────────────────────────
import torch
print(f'[verify] torch={torch.__version__} cuda={torch.cuda.is_available()}', flush=True)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[verify] GPU: {gpu} ({vram:.1f} GB)', flush=True)
    if vram < 14:
        print(f'[verify] WARNING: only {vram:.0f} GB VRAM. SuGaR will likely OOM.', flush=True)
        print(f'[verify]   Recommended: L4 (22 GB) or A100 (40 GB).', flush=True)
    elif vram < 20:
        print(f'[verify] VRAM is tight. Will use --low_poly True --refinement_time short.', flush=True)
    else:
        print(f'[verify] VRAM is comfortable for the full SuGaR pipeline.', flush=True)
try:
    import pytorch3d
    print(f'[verify] pytorch3d={pytorch3d.__version__} OK', flush=True)
except ImportError as e:
    print(f'[verify] pytorch3d FAILED: {e}', flush=True)
    raise
try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer
    print(f'[verify] diff_gaussian_rasterization OK', flush=True)
except ImportError as e:
    print(f'[verify] diff_gaussian_rasterization FAILED: {e}', flush=True)
    print(f'[verify] Re-run the build loop in this cell after checking the error above.', flush=True)
try:
    from simple_knn._C import distCUDA2
    print(f'[verify] simple_knn OK', flush=True)
except ImportError as e:
    print(f'[verify] simple_knn FAILED: {e}', flush=True)
try:
    import open3d, trimesh, plyfile, pymcubes
    print(f'[verify] open3d={open3d.__version__} trimesh={trimesh.__version__} OK', flush=True)
except ImportError as e:
    print(f'[verify] mesh deps FAILED: {e}', flush=True)

print(f'\n[Done] STEP 1 complete. Move on to STEP 2 (TripoSplat-PLY bridge).', flush=True)


In [ ]:
# @title STEP 2 — TripoSplat-PLY-to-3DGS-Scene Bridge (single-image input)
# @markdown SuGaR's design assumes multi-view input (COLMAP cameras + trained
# @markdown 3DGS checkpoint). For a **single image** + TripoSplat's 3DGS output
# @markdown PLY, we need to build a minimal COLMAP-compatible scene directory
# @markdown with one camera. This is non-trivial — SuGaR's input contract is
# @markdown strict. We do the best we can:
# @markdown   1. Use TripoSplat's preprocessing to estimate the camera intrinsics
# @markdown      (BiRefNet BG removal + 1024×1024 crop gives a known FoV).
# @markdown   2. Place the camera at the origin looking down the +Z axis,
# @markdown      matching TripoSplat's canonical [-0.5, 0.5]³ aabb.
# @markdown   3. Write the 3DGS PLY into the 3DGS-trainable checkpoint structure.
# @markdown   4. Initialize the 3DGS model from TripoSplat's PLY (skip 7k iters
# @markdown      of vanilla training, save 30 min).
#
# @markdown **Quality caveat:** for single-image input, this bridge is the best
# @markdown we can do. The mesh quality gain over TripoSplat's default Poisson
# @markdown is **modest** (no multi-view consistency signal). The real SuGaR
# @markdown benefit kicks in for **multi-image** inputs (you'd skip TripoSplat
# @markdown and run SuGaR directly on a multi-view 3DGS scene).

import os, sys, time, shutil, json, pathlib
import numpy as np
import torch

# Where to find the TripoSplat output
TRIPOSPLAT_PLY = '/content/triposplat_runs/quicktest/quick.ply'  # @param {type:"string"}
# @markdown *Path to a TripoSplat-generated .ply (the 3DGS Gaussian PLY, NOT the _mesh.ply).*
TRIPOSPLAT_IMAGE = '/content/triposplat_runs/quicktest/preprocessed.png'  # @param {type:"string"}
# @markdown *Path to the preprocessed 1024×1024 image TripoSplat fed to the model. Used for camera intrinsics + scene input image.*
SCENE_NAME = 'triposplat_bridge'  # @param {type:"string"}
# @markdown *Output scene name. Will be created under /content/SuGaR/data/<name>/.*
SUGAR_REPO = '/content/SuGaR'
SCENE_DIR = os.path.join(SUGAR_REPO, 'data', SCENE_NAME)
GS_OUTPUT_DIR = os.path.join(SUGAR_REPO, 'output', 'vanilla_gs', SCENE_NAME)

ply_path = pathlib.Path(TRIPOSPLAT_PLY)
img_path = pathlib.Path(TRIPOSPLAT_IMAGE)
if not ply_path.exists():
    raise SystemExit(f'[bridge] TripoSplat PLY not found: {ply_path}\n'
                      f'Run TripoSplat_Colab.ipynb Step 6 first.')
if not img_path.exists():
    raise SystemExit(f'[bridge] Preprocessed image not found: {img_path}')
print(f'[bridge] TripoSplat PLY: {ply_path} ({ply_path.stat().st_size/1024/1024:.1f} MB)')
print(f'[bridge] Preprocessed image: {img_path}')

# ── Build COLMAP scene structure ──────────────────────────────────────
# COLMAP needs: images/<name>.jpg, sparse/0/cameras.txt, sparse/0/images.txt,
# sparse/0/points3D.txt. For 1 image we make 1 camera + 1 image + empty
# points3D (the 3DGS init replaces this).
for sub in ('images', 'sparse/0'):
    os.makedirs(os.path.join(SCENE_DIR, sub), exist_ok=True)

# Copy the image to images/ (3DGS reads it from here)
import shutil
img_dst = os.path.join(SCENE_DIR, 'images', img_path.name)
if not os.path.exists(img_dst):
    shutil.copy2(img_path, img_dst)
    print(f'[bridge] Copied image to {img_dst}')

# ── Camera intrinsics (estimated from TripoSplat's preprocessing) ────────────────
# TripoSplat preprocesses to 1024x1024, BiRefNet BG removal, crop + resize.
# We use a generic perspective camera with ~50° FoV (typical for portrait/object
# shots). The intrinsics matrix is approximate but enough for 3DGS to start
# training — 3DGS learns camera refinement during the 7k iterations.
from PIL import Image
with Image.open(img_path) as im:
    W, H = im.size
# ~50° vertical FoV, principal point at center
fov_y_deg = 50.0
fy = (H / 2.0) / np.tan(np.deg2rad(fov_y_deg / 2.0))
fx = fy  # square pixels
cx, cy = W / 2.0, H / 2.0
cameras_txt = (
    '# Camera list with one line of data per camera:\n'
    '#   CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n'
    f'1 PINHOLE {W} {H} {fx:.6f} {fy:.6f} {cx:.6f} {cy:.6f}\n'
)
with open(os.path.join(SCENE_DIR, 'sparse/0', 'cameras.txt'), 'w') as f:
    f.write(cameras_txt)
print(f'[bridge] Camera: PINHOLE {W}x{H} fx={fx:.1f} fy={fy:.1f} cx={cx:.1f} cy={cy:.1f} (FoV_y={fov_y_deg}°)')

# ── Single image (camera pose = identity — looking down -Z, world origin at subject) ──
# 3DGS convention: +Z is forward, +Y is up, +X is right. We place the camera
# at (0, 0, 2.5) looking at origin — subject occupies [-0.5, 0.5]³.
# SuGaR will refine this during training, so the exact value doesn't matter
# much.
import numpy as np
# Camera position: 2.5 units back from origin, looking at -Z
cam_pos = np.array([0.0, 0.0, 2.5])
# Look-at matrix: forward = -Z (toward origin), up = +Y
forward = -cam_pos / np.linalg.norm(cam_pos)  # toward origin
world_up = np.array([0.0, 1.0, 0.0])
right = np.cross(forward, world_up)
right /= np.linalg.norm(right)
up = np.cross(right, forward)
# COLMAP quaternion: (qw, qx, qy, qz). Rotation from world to camera = camera basis.
R = np.stack([right, up, -forward], axis=0)  # 3x3, columns = camera basis in world
from scipy.spatial.transform import Rotation
quat = Rotation.from_matrix(R.T).as_quat()  # (x, y, z, w) -> reorder to (w, x, y, z)
qvec = f'{quat[3]:.9f} {quat[0]:.9f} {quat[1]:.9f} {quat[2]:.9f}'
tvec = f'{-R.T @ cam_pos[0]:.9f} {-R.T @ cam_pos[1]:.9f} {-R.T @ cam_pos[2]:.9f}'
images_txt = (
    '# Image list with two lines of data per image:\n'
    '#   IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n'
    '#   POINTS2D[] as (X, Y, POINT3D_ID)\n'
    f'1 {qvec} {tvec} 1 {img_path.name}\n\n'
)
with open(os.path.join(SCENE_DIR, 'sparse/0', 'images.txt'), 'w') as f:
    f.write(images_txt)
print(f'[bridge] Image pose: pos={tuple(cam_pos.round(3))} (SuGaR will refine this during training)')

# ── Empty points3D.txt (3DGS inits from its own random Gaussians) ──
with open(os.path.join(SCENE_DIR, 'sparse/0', 'points3D.txt'), 'w') as f:
    f.write('# 3D point list with one line of data per point:\n'
            '#   POINT3D_ID, X, Y, Z, R, G, B, ERROR, TRACK[]\n')
print(f'[bridge] Empty points3D.txt (3DGS inits from random Gaussians)')

# ── Also create a 3DGS-trainable checkpoint from TripoSplat's PLY ────────────
# 3DGS expects: point_cloud/iteration_<N>/point_cloud.ply + cameras.json +
# cfg_args. We'll convert TripoSplat's PLY format to 3DGS format.
#
# TripoSplat PLY schema (3DGS-extended): x,y,z, nx,ny,nz, f_dc_0/1/2 (SH0),
# f_rest_0..44 (SH1-3), opacity, scale_0/1/2, rot_0/1/2/3 (quaternion w,x,y,z)
#
# This is actually the SAME schema as vanilla 3DGS-trained PLYs! TripoSplat
# uses 3DGS conventions for its output. So we can just copy the PLY into
# the right place and write minimal cameras.json + cfg_args.
os.makedirs(os.path.join(GS_OUTPUT_DIR, 'point_cloud', 'iteration_7000'), exist_ok=True)
gs_ply = os.path.join(GS_OUTPUT_DIR, 'point_cloud', 'iteration_7000', 'point_cloud.ply')
shutil.copy2(ply_path, gs_ply)
print(f'[bridge] Copied 3DGS PLY to {gs_ply}')

cameras_json = {
    'W': W, 'H': H, 'fx': fx, 'fy': fy, 'cx': cx, 'cy': cy,
    'num_images': 1,
    'image_names': [img_path.name],
    'poses': [{'qvec': qvec.split(), 'tvec': tvec.split()}],
}
with open(os.path.join(GS_OUTPUT_DIR, 'cameras.json'), 'w') as f:
    json.dump(cameras_json, f, indent=2)
cfg_args = {
    'sh_degree': 3,
    'source_path': SCENE_DIR,
    'model_path': GS_OUTPUT_DIR,
    'images': 'images',
    'resolution': -1,
    'white_background': False,
    'data_device': 'cuda',
    'eval': False,
    'shs_degree': 0,
}
with open(os.path.join(GS_OUTPUT_DIR, 'cfg_args'), 'w') as f:
    json.dump(cfg_args, f, indent=2)
print(f'[bridge] Wrote cameras.json + cfg_args')

print(f'\n[Done] Bridge complete. Scene at: {SCENE_DIR}')
print(f'[Done] 3DGS checkpoint at: {GS_OUTPUT_DIR}')
print(f'[Done] Move on to STEP 3 (vanilla 3DGS training — optional but recommended).')


In [ ]:
# @title STEP 3 — Vanilla 3DGS Training (Optional, ~30 min on L4)
# @markdown SuGaR's surface-alignment works best when the Gaussians are already
# @markdown optimized via vanilla 3DGS training. We **can** skip this step if you
# @markdown initialized the scene from TripoSplat's PLY (Step 2 already wrote a
# @markdown 3DGS checkpoint), but **running 7k iters of vanilla 3DGS usually gives
# @markdown a better surface** for SuGaR to align to.
#
# @markdown This step runs SuGaR's bundled vanilla 3DGS trainer
# @markdown (`gaussian_splatting/train.py`) against the COLMAP scene we built.
# @markdown First run: ~5 min for COLMAP + 30 min for 3DGS on L4.
#
# @markdown **Skip if** you initialized from TripoSplat's PLY AND are OK with
# @markdown marginally worse mesh quality. The PLY already encodes TripoSplat's
# @markdown learned Gaussians, so vanilla 3DGS just re-trains them.

SKIP_3DGS_TRAIN = True  # @param {type:"boolean"}
# @markdown *Set to True if you initialized from TripoSplat's PLY (Step 2) and want to skip the 30-min re-train. False to run 7k iters of vanilla 3DGS.*
ITERATIONS = 7000  # @param {type:"slider", min:1000, max:30000, step:1000}
# @markdown *3DGS iterations. 7000 is the SuGaR default. 30000 = better quality, ~2x time.*
DENSITY_REG = 'dn_consistency'  # @param ["dn_consistency", "densify", "none"]
# @markdown *Density regularization. dn_consistency is SuGaR's default; densify = vanilla 3DGS.*
EVAL = False  # @param {type:"boolean"}
# @markdown *Run eval split (held-out test cameras). We only have 1 image so this is moot.*

import os, sys, time, subprocess, pathlib
SUGAR_REPO = '/content/SuGaR'
SCENE_NAME = 'triposplat_bridge'
SCENE_DIR = os.path.join(SUGAR_REPO, 'data', SCENE_NAME)
GS_OUTPUT_DIR = os.path.join(SUGAR_REPO, 'output', 'vanilla_gs', SCENE_NAME)
os.chdir(SUGAR_REPO)

if SKIP_3DGS_TRAIN:
    print('[3dgs] Skipping vanilla 3DGS training (using TripoSplat-init checkpoint)')
    print('[3dgs] Move on to STEP 4 (SuGaR surface-alignment).')
    print(flush=True)
else:
    t0 = time.time()
    print(f'[3dgs] Training vanilla 3DGS for {ITERATIONS} iters (~30-60 min on L4)...', flush=True)
    print(f'[3dgs] Scene: {SCENE_DIR}', flush=True)
    print(f'[3dgs] Output: {GS_OUTPUT_DIR}', flush=True)
    cmd = [
        sys.executable, 'gaussian_splatting/train.py',
        '-s', SCENE_DIR,
        '-m', GS_OUTPUT_DIR,
        '--iterations', str(ITERATIONS),
        '--eval', str(EVAL),
    ]
    print(f'[3dgs] CMD: {" ".join(cmd)}', flush=True)
    # Run with real-time output streaming
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 universal_newlines=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise SystemExit(f'[3dgs] Training failed (returncode={process.returncode})')
    print(f'\n[3dgs] Training complete in {(time.time()-t0)/60:.1f} min', flush=True)
    print(f'[3dgs] Checkpoint at: {GS_OUTPUT_DIR}/point_cloud/iteration_{ITERATIONS}/point_cloud.ply', flush=True)
    print(f'[3dgs] Move on to STEP 4 (SuGaR surface-alignment).', flush=True)


In [ ]:
# @title STEP 4 — SuGaR Surface-Alignment + Mesh Extraction (~1-2 hrs on L4)
# @markdown The main SuGaR step. Takes the 3DGS checkpoint (from TripoSplat's
# @markdown PLY init or from Step 3's vanilla train), adds surface-aligned
# @markdown constraints, refines the Gaussians to lie on a coherent surface, and
# @markdown extracts a textured mesh.
#
# @markdown **Time on L4:** ~60-90 min for low-poly + short refinement, ~2-3 hrs
# @markdown for high-poly + long refinement. T4 will likely OOM.

LOW_POLY = True  # @param {type:"boolean"}
# @markdown *low_poly = ~200k vertices, ~6 Gaussians/triangle (memory-efficient). high_poly = ~1M vertices, 1 Gaussian/tri (best quality, needs A100).*
REFINEMENT_TIME = 'short'  # @param ["short", "medium", "long"]
# @markdown *short = 2k iters (~30 min). medium = 7k iters (~1.5 hrs). long = 15k iters (~3 hrs).*
EXPORT_OBJ = True  # @param {type:"boolean"}
EXPORT_PLY = True  # @param {type:"boolean"}
# @markdown *Export textured .obj + texture atlas + refined 3DGS .ply. Always leave both True.*
GPU = 0  # @param {type:"integer"}
# @markdown *GPU index. Always 0 in Colab.*

import os, sys, time, subprocess, pathlib
SUGAR_REPO = '/content/SuGaR'
SCENE_NAME = 'triposplat_bridge'
SCENE_DIR = os.path.join(SUGAR_REPO, 'data', SCENE_NAME)
GS_OUTPUT_DIR = os.path.join(SUGAR_REPO, 'output', 'vanilla_gs', SCENE_NAME)
SUGAR_OUTPUT_DIR = os.path.join(SUGAR_REPO, 'output', SCENE_NAME)
os.makedirs(SUGAR_OUTPUT_DIR, exist_ok=True)
os.chdir(SUGAR_REPO)

if not os.path.exists(os.path.join(GS_OUTPUT_DIR, 'cameras.json')):
    raise SystemExit(f'[sugar] 3DGS checkpoint not found at {GS_OUTPUT_DIR}. Run STEP 2 + 3 first.')

t0 = time.time()
print(f'[sugar] Running SuGaR surface-alignment + mesh extraction...', flush=True)
print(f'[sugar] low_poly={LOW_POLY} refinement={REFINEMENT_TIME}', flush=True)
print(f'[sugar] This will take 1-3 hours. Do NOT interrupt.', flush=True)
print(flush=True)

cmd = [
    sys.executable, 'train.py',
    '-s', SCENE_DIR,
    '-c', GS_OUTPUT_DIR,
    '-r', 'dn_consistency',
    '-f', '2000',  # 2k iters of refinement (will be capped by --refinement_time)
    '--low_poly', str(LOW_POLY),
    '--refinement_time', REFINEMENT_TIME,
    '--export_obj', str(EXPORT_OBJ),
    '--export_ply', str(EXPORT_PLY),
    '--eval', 'False',
    '--gpu', str(GPU),
]
print(f'[sugar] CMD: {" ".join(cmd)}', flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             universal_newlines=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
if process.returncode != 0:
    raise SystemExit(f'[sugar] SuGaR failed (returncode={process.returncode})')
print(f'\n[sugar] SuGaR complete in {(time.time()-t0)/60:.1f} min', flush=True)
print(f'[sugar] Outputs should be in: {SUGAR_OUTPUT_DIR}/refined_mesh/', flush=True)
print(f'[sugar] Move on to STEP 5 to verify + copy to Drive.', flush=True)


In [ ]:
# @title STEP 5 — Verify Outputs + Game-Asset Transform
# @markdown Lists the SuGaR outputs, applies a game-asset transform (scale to
# @markdown meters, base-pivot center), and converts the .obj to .glb for
# @markdown game engines that prefer glTF (Three.js, model-viewer, Unity).

import os, sys, time, pathlib, shutil
import numpy as np
import trimesh
import open3d as o3d
from PIL import Image

SUGAR_REPO = '/content/SuGaR'
SCENE_NAME = 'triposplat_bridge'
SUGAR_OUTPUT_DIR = os.path.join(SUGAR_REPO, 'output', SCENE_NAME)
REFINED_MESH_DIR = os.path.join(SUGAR_OUTPUT_DIR, 'refined_mesh')
REFINED_PLY_DIR = os.path.join(SUGAR_OUTPUT_DIR, 'refined_ply')
COARSE_MESH_DIR = os.path.join(SUGAR_OUTPUT_DIR, 'coarse_mesh')
GAME_OUT_DIR = '/content/sugar_game_out'
os.makedirs(GAME_OUT_DIR, exist_ok=True)

# ── Locate the textured .obj ────────────────────────────────────────────────
if not os.path.isdir(REFINED_MESH_DIR):
    print(f'[verify] WARNING: {REFINED_MESH_DIR} not found.')
    print(f'[verify]   Did STEP 4 complete? Check the logs above for errors.')
    print(f'[verify]   Looking for any .obj in {SUGAR_OUTPUT_DIR}...')
    objs = list(pathlib.Path(SUGAR_OUTPUT_DIR).rglob('*.obj'))
    if objs:
        print(f'[verify]   Found {len(objs)} .obj files: {[str(p) for p in objs[:5]]}')
        TEXTURED_OBJ = objs[0]
        REFINED_MESH_DIR = str(TEXTURED_OBJ.parent)
    else:
        raise SystemExit(f'[verify] No .obj files found. STEP 4 may have failed.')
else:
    objs = sorted(pathlib.Path(REFINED_MESH_DIR).rglob('*.obj'))
    if not objs:
        raise SystemExit(f'[verify] No .obj in {REFINED_MESH_DIR}')
    TEXTURED_OBJ = objs[0]
TEXTURED_OBJ = str(TEXTURED_OBJ)
print(f'[verify] Textured mesh: {TEXTURED_OBJ}')
# Show associated files (.mtl + texture png)
for p in pathlib.Path(TEXTURED_OBJ).parent.iterdir():
    print(f'  {p.name}: {p.stat().st_size/1024:.0f} KB')

# ── Game-asset transform (scale to meters, base-pivot center) ──────────────
TARGET_SIZE = 1.0  # @param {type:"slider", min:0.05, max:5.0, step:0.05}
# @markdown *Longest-edge target in meters. 0.3=toy, 1.0=human, 1.8=vehicle, 2.5=building.*
CENTER_MODE = 'base'  # @param ["base", "origin", "keep"]
# @markdown *base = bbox bottom at origin (actor pivot). origin = center at origin. keep = no transform.*

print(f'[game] Loading {TEXTURED_OBJ} for game-asset transform...')
mesh = trimesh.load(TEXTURED_OBJ, force='mesh', process=False)
n_v0, n_f0 = len(mesh.vertices), len(mesh.faces)
print(f'[game] Loaded: {n_v0:,} verts / {n_f0:,} faces')

if TARGET_SIZE > 0 and CENTER_MODE != 'keep':
    bbox_min = mesh.vertices.min(axis=0)
    bbox_max = mesh.vertices.max(axis=0)
    bbox_center = (bbox_min + bbox_max) / 2.0
    bbox_extent = float((bbox_max - bbox_min).max())
    if bbox_extent > 1e-9:
        scale = TARGET_SIZE / bbox_extent
    else:
        scale = 1.0
    if CENTER_MODE == 'origin':
        translate = -bbox_center
    elif CENTER_MODE == 'base':
        translate = -np.array([bbox_center[0], bbox_min[1], bbox_center[2]])
    new_verts = (mesh.vertices + translate) * scale
    mesh.vertices = new_verts
    print(f'[game] Transformed: scale={scale:.3f} translate={translate.round(3).tolist()}')
    print(f'[game] New bbox extent: {(mesh.vertices.max(axis=0) - mesh.vertices.min(axis=0)).max():.3f}m')

# ── Save transformed mesh ────────────────────────────────────────────────
game_obj = os.path.join(GAME_OUT_DIR, f'{SCENE_NAME}_game.obj')
game_glb = os.path.join(GAME_OUT_DIR, f'{SCENE_NAME}_game.glb')
game_ply = os.path.join(GAME_OUT_DIR, f'{SCENE_NAME}_game.ply')

# Export .obj (preserves UVs + texture reference)
mesh.export(game_obj, file_type='obj')
print(f'[game] Saved: {game_obj}')
# Copy the .mtl + texture PNG if present
for ext in ('.mtl', '_textured.png', '.png', '_albedo.png', '_diffuse.png'):
    for src in pathlib.Path(TEXTURED_OBJ).parent.glob(f'*{ext}'):
        if src.name == pathlib.Path(TEXTURED_OBJ).name:
            continue
        dst = os.path.join(GAME_OUT_DIR, f'{SCENE_NAME}_game{ext}')
        shutil.copy2(src, dst)
        print(f'[game] Copied: {dst}')

# Export .glb (single-file, texture embedded)
try:
    mesh_for_glb = trimesh.load(TEXTURED_OBJ, force='mesh', process=False)
    if TARGET_SIZE > 0 and CENTER_MODE != 'keep':
        mesh_for_glb.vertices = (mesh_for_glb.vertices + translate) * scale
    mesh_for_glb.export(game_glb, file_type='glb')
    print(f'[game] Saved: {game_glb}')
except Exception as e:
    print(f'[game] GLB export failed: {e} (obj should still work)')

# Export .ply (debug, smaller)
mesh.export(game_ply, file_type='ply')
print(f'[game] Saved: {game_ply}')

print(f'\n[Done] Game-ready mesh in {GAME_OUT_DIR}', flush=True)
print(f'[Done] Move on to STEP 7 to mirror to Drive.')


In [ ]:
# @title STEP 6 — Copy to Drive
# @markdown Mirrors the game-ready mesh + SuGaR raw outputs to your Drive at
# @markdown `/content/drive/MyDrive/AEI_3D_Out/SuGaR/<scene_name>/`. Skipped if
# @markdown Drive is not mounted.

import os, shutil, pathlib
from IPython.display import FileLink, display

SCENE_NAME = 'triposplat_bridge'
SUGAR_OUTPUT_DIR = f'/content/SuGaR/output/{SCENE_NAME}'
GAME_OUT_DIR = '/content/sugar_game_out'
DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/SuGaR')

if not DRIVE_BASE.parent.exists():
    print('[drive] /content/drive not mounted. Skipping Drive mirror.')
    print('[drive]   Mount Drive first: from the Colab file browser, click the Drive icon.')
else:
    # Game-ready output (the one you actually want to use)
    game_dst = DRIVE_BASE / 'game_ready' / SCENE_NAME
    game_dst.mkdir(parents=True, exist_ok=True)
    if os.path.isdir(GAME_OUT_DIR):
        for f in pathlib.Path(GAME_OUT_DIR).iterdir():
            if f.is_file():
                shutil.copy2(f, game_dst / f.name)
        n = sum(1 for _ in game_dst.iterdir())
        print(f'[drive] Mirrored {n} game-ready files to {game_dst}')
    # Full SuGaR raw outputs (for debugging or re-processing)
    raw_dst = DRIVE_BASE / 'raw_sugar' / SCENE_NAME
    if os.path.isdir(SUGAR_OUTPUT_DIR):
        raw_dst.mkdir(parents=True, exist_ok=True)
        # Don't copy the massive intermediate .pt files (~500 MB each)
        for f in pathlib.Path(SUGAR_OUTPUT_DIR).rglob('*'):
            if f.is_file() and not f.name.endswith('.pt') and f.stat().st_size < 200_000_000:
                rel = f.relative_to(SUGAR_OUTPUT_DIR)
                dst = raw_dst / rel
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)
        n = sum(1 for _ in raw_dst.rglob('*') if _.is_file())
        print(f'[drive] Mirrored {n} raw SuGaR files to {raw_dst}')
    # Show download links for the game-ready files
    print('\n[drive] Download links:')
    if os.path.isdir(GAME_OUT_DIR):
        for f in sorted(pathlib.Path(GAME_OUT_DIR).iterdir()):
            if f.is_file():
                display(FileLink(str(f)))
                print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f} MB)')


In [ ]:
# @title STEP 7 — Keep-Alive (anti-disconnect)
# @markdown SuGaR + 3DGS training takes 2-3 hours. Run this cell to prevent
# @markdown Colab from disconnecting you mid-training. Standard pattern across
# @markdown the suite.

import IPython
from google.colab import output
KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""
display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started — pings every 30 min. Run this in a separate cell from the SuGaR training.')


In [ ]:
# @title STEP 8 — Help / License / Known Issues
# @markdown Reference material. Read this if anything in the pipeline went wrong.

help_md = '''
# SuGaR — Help, License, and Known Issues

## License (READ THIS)

**SuGaR uses the [INRIA Gaussian-Splatting License](https://github.com/Anttwo/SuGaR/blob/main/LICENSE.md) (custom non-commercial research license):**

- ✅ Free for academic and industrial **research / evaluation** use
- ❌ **NOT for commercial use** without explicit INRIA consent (`stip-sophia.transfert@inria.fr`)
- Derivative works must keep this license or apply Your Terms only to the derivative portions
- Redistribution must include the full LICENSE.md + attribution
- **Required citation:** Guédon & Lepetit, "SuGaR: Surface-Aligned Gaussian Splatting for 3D Mesh Reconstruction", CVPR 2024

**If you ship a commercial product:** you must (a) get INRIA's permission, (b) remove SuGaR and use a commercial alternative ([Kiri Engine](https://www.kiriengine.app/blog/what-is-3dgs-to-mesh), [Polycam](https://poly.cam/), [LumaGen](https://lumalabs.ai/gen)), or (c) train your own surface-aligned Gaussian method.

## Pipeline overview

```
TripoSplat (image → 3DGS PLY)  →  Step 2 bridge (PLY → COLMAP scene)  →  Step 3 (optional 3DGS train)  →  Step 4 (SuGaR align + mesh)  →  Step 5 (game-asset transform + .glb)  →  Step 6 (Drive)
```

## Compute

| GPU | VRAM | Time (low_poly + short) | Time (high_poly + long) |
|-----|------|-------------------------|-------------------------|
| T4  | 16 GB | Likely OOM | OOM |
| L4  | 22 GB | ~1.5 hrs | ~3 hrs |
| A100 | 40 GB | ~1 hr | ~2 hrs |
| A100 | 80 GB | ~45 min | ~1.5 hrs |

## Known issues

1. **`ImportError: libc10_cuda.so`** — PyTorch / pytorch3d / CUDA version mismatch. We pin `torch==2.0.1+cu118` and use the matching pytorch3d wheel. If this still fails, fall back to `conda install pytorch3d=0.7.4 -c pytorch3d`.
2. **`diff-gaussian-rasterization` build failure** — the submodules are CUDA extensions that must compile against the active PyTorch. The build loop in STEP 1 prints the error. Common cause: someone else updated the GPU driver. Fix: re-run STEP 1.
3. **Single-image mesh quality is modest** — SuGaR's surface constraint has limited information from one view. The mesh will be better than TripoSplat's default Poisson, but not dramatically so. **The real SuGaR benefit is for multi-view inputs** (3+ overlapping photos of the same subject).
4. **For 200+ single-view subjects**, the honest answer is that no open-source tool will give you game-ready meshes. Your best options are:
   - **Use the 3DGS PLY directly** in a 3DGS-aware viewer (Antimatter15, gsplat.js, LumaAI) — best visual quality, no mesh needed
   - **Use SuGaR on your 5-10 hero assets** where the extra quality is worth 2 hrs/scene
   - **Use a commercial tool** (Kiri Engine, Polycam) for the long tail
   - **Accept TripoSplat's mesh as a low-LOD fallback** for the rest

## Citation

```bibtex
@InProceedings{Guedon_2024_CVPR,
    author    = {Guédon, Antoine and Lepetit, Vincent},
    title     = {{SuGaR}: Surface-Aligned Gaussian Splatting for 3D Mesh Reconstruction},
    booktitle = {CVPR},
    year      = {2024},
}
```
'''
from IPython.display import Markdown, display
display(Markdown(help_md))
print(help_md)
